In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/netflix-prize-data/combined_data_3.txt
/kaggle/input/netflix-prize-data/movie_titles.csv
/kaggle/input/netflix-prize-data/combined_data_4.txt
/kaggle/input/netflix-prize-data/combined_data_1.txt
/kaggle/input/netflix-prize-data/README
/kaggle/input/netflix-prize-data/probe.txt
/kaggle/input/netflix-prize-data/combined_data_2.txt
/kaggle/input/netflix-prize-data/qualifying.txt


In [21]:
with open('/kaggle/input/netflix-prize-data/combined_data_1.txt') as f:
    for i in range(20):
        line = f.readline()
        print(f'Line {i+1} :  {line.strip()}')

Line 1 :  1:
Line 2 :  1488844,3,2005-09-06
Line 3 :  822109,5,2005-05-13
Line 4 :  885013,4,2005-10-19
Line 5 :  30878,4,2005-12-26
Line 6 :  823519,3,2004-05-03
Line 7 :  893988,3,2005-11-17
Line 8 :  124105,4,2004-08-05
Line 9 :  1248029,3,2004-04-22
Line 10 :  1842128,4,2004-05-09
Line 11 :  2238063,3,2005-05-11
Line 12 :  1503895,4,2005-05-19
Line 13 :  2207774,5,2005-06-06
Line 14 :  2590061,3,2004-08-12
Line 15 :  2442,3,2004-04-14
Line 16 :  543865,4,2004-05-28
Line 17 :  1209119,4,2004-03-23
Line 18 :  804919,4,2004-06-10
Line 19 :  1086807,3,2004-12-28
Line 20 :  1711859,4,2005-05-08


In [22]:
curr_movie_id = None
data = []
import pandas as pd
with open('/kaggle/input/netflix-prize-data/combined_data_1.txt') as f:
    for i,line in enumerate(f):
        line = line.strip()
       
        if line.endswith(':'):
            curr_movie_id = int(line[:-1])
        else:
            customer_id,rating,review_date = line.split(",")
            data.append({
                'movie_id' : curr_movie_id,
                'customer_id' : int(customer_id),
                'rating' : int(rating),x
                'review_date' : review_date
                
                
            })
        if i >= 1000:
            break
df_test = pd.DataFrame(data)
df_test.head(200)
            
        

  


,movie_id,customer_id,rating,review_date
0,1,1488844,3,2005-09-06
1,1,822109,5,2005-05-13
2,1,885013,4,2005-10-19
3,1,30878,4,2005-12-26
4,1,823519,3,2004-05-03
...,...,...,...,...
195,1,352635,5,2004-05-11
196,1,2537543,5,2005-10-26
197,1,1564395,4,2005-07-09
198,1,1655178,4,2004-08-26


In [23]:
df_test['movie_id'].unique()


array([1, 2, 3])

In [24]:
from pyspark.sql import SparkSession 

spark = SparkSession.builder.appName('NetflixRecommendation').getOrCreate()

In [25]:
from pyspark.sql.types import StructType, StructField, IntegerType,StringType

schema = StructType([
    StructField('movie_id',IntegerType(),True),
    StructField('customer_id', IntegerType(),True),
    StructField('rating', IntegerType(),True),
    StructField('review_date', StringType(),True),

    
    
])

In [28]:
# Create Spark DataFrame
df_ratings = spark.createDataFrame(data[:1001], schema=schema)

# Show first 10 rows
df_ratings.show(10)

# Check the schema
df_ratings.printSchema()

# Count total rows
print(f"Total rows in DataFrame: {df_ratings.count()}")

+--------+-----------+------+-----------+
|movie_id|customer_id|rating|review_date|
+--------+-----------+------+-----------+
|       1|    1488844|     3| 2005-09-06|
|       1|     822109|     5| 2005-05-13|
|       1|     885013|     4| 2005-10-19|
|       1|      30878|     4| 2005-12-26|
|       1|     823519|     3| 2004-05-03|
|       1|     893988|     3| 2005-11-17|
|       1|     124105|     4| 2004-08-05|
|       1|    1248029|     3| 2004-04-22|
|       1|    1842128|     4| 2004-05-09|
|       1|    2238063|     3| 2005-05-11|
+--------+-----------+------+-----------+
only showing top 10 rows

root
 |-- movie_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_date: string (nullable = true)

Total rows in DataFrame: 998


In [29]:
df_ratings.show(3)

+--------+-----------+------+-----------+
|movie_id|customer_id|rating|review_date|
+--------+-----------+------+-----------+
|       1|    1488844|     3| 2005-09-06|
|       1|     822109|     5| 2005-05-13|
|       1|     885013|     4| 2005-10-19|
+--------+-----------+------+-----------+
only showing top 3 rows



In [30]:
def parse_netflix_file(filepath, num_records=1000):
    """
    Parse a Netflix combined_data file and return first num_records
    
    Args:
        filepath: Path to the file
        num_records: Number of records to parse
    
    Returns:
        pandas DataFrame with parsed data
    """
    curr_movie_id = None
    data = []
    
    with open(filepath) as f:
        for i, line in enumerate(f):
            line = line.strip()
           
            if line.endswith(':'):
                curr_movie_id = int(line[:-1])
            else:
                customer_id, rating, review_date = line.split(",")
                data.append({
                    'movie_id': curr_movie_id,
                    'customer_id': int(customer_id),
                    'rating': int(rating),
                    'review_date': review_date
                })
            
            if len(data) >= num_records:  # Use len(data) not i
                break
    
    return pd.DataFrame(data)

# Now use it for all 4 files
df1 = parse_netflix_file('/kaggle/input/netflix-prize-data/combined_data_1.txt', 1000)
df2 = parse_netflix_file('/kaggle/input/netflix-prize-data/combined_data_2.txt', 1000)
df3 = parse_netflix_file('/kaggle/input/netflix-prize-data/combined_data_3.txt', 1000)
df4 = parse_netflix_file('/kaggle/input/netflix-prize-data/combined_data_4.txt', 1000)

print(f"df1 shape: {df1.shape}")
print(f"df2 shape: {df2.shape}")
print(f"df3 shape: {df3.shape}")
print(f"df4 shape: {df4.shape}")

df1 shape: (1000, 4)
df2 shape: (1000, 4)
df3 shape: (1000, 4)
df4 shape: (1000, 4)


In [31]:
df1_ratings = spark.createDataFrame(df1, schema=schema)

# Show first 10 rows
df1_ratings.show(10)

# Check the schema
df1_ratings.printSchema()

# Count total rows
print(f"Total rows in DataFrame: {df1_ratings.count()}")

+--------+-----------+------+-----------+
|movie_id|customer_id|rating|review_date|
+--------+-----------+------+-----------+
|       1|    1488844|     3| 2005-09-06|
|       1|     822109|     5| 2005-05-13|
|       1|     885013|     4| 2005-10-19|
|       1|      30878|     4| 2005-12-26|
|       1|     823519|     3| 2004-05-03|
|       1|     893988|     3| 2005-11-17|
|       1|     124105|     4| 2004-08-05|
|       1|    1248029|     3| 2004-04-22|
|       1|    1842128|     4| 2004-05-09|
|       1|    2238063|     3| 2005-05-11|
+--------+-----------+------+-----------+
only showing top 10 rows

root
 |-- movie_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_date: string (nullable = true)

Total rows in DataFrame: 1000


In [32]:
df2_ratings = spark.createDataFrame(df2, schema=schema)
df3_ratings = spark.createDataFrame(df2, schema=schema)
df4_ratings = spark.createDataFrame(df2, schema=schema)




In [35]:
df_all = df1_ratings.union(df2_ratings).union(df3_ratings).union(df4_ratings)

print(f"Total rows after merging: {df_all.count()}")
print("\nSchema:")
df_all.printSchema()
print("\nFirst 10 rows:")
df_all.show(10)

Total rows after merging: 4000

Schema:
root
 |-- movie_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_date: string (nullable = true)


First 10 rows:
+--------+-----------+------+-----------+
|movie_id|customer_id|rating|review_date|
+--------+-----------+------+-----------+
|       1|    1488844|     3| 2005-09-06|
|       1|     822109|     5| 2005-05-13|
|       1|     885013|     4| 2005-10-19|
|       1|      30878|     4| 2005-12-26|
|       1|     823519|     3| 2004-05-03|
|       1|     893988|     3| 2005-11-17|
|       1|     124105|     4| 2004-08-05|
|       1|    1248029|     3| 2004-04-22|
|       1|    1842128|     4| 2004-05-09|
|       1|    2238063|     3| 2005-05-11|
+--------+-----------+------+-----------+
only showing top 10 rows



In [36]:
# Read movie_titles.csv into Spark DataFrame
df_movies = spark.read.csv(
    '/kaggle/input/netflix-prize-data/movie_titles.csv',
    header=False,  # This file has no header row
    inferSchema=True,
    encoding='ISO-8859-1'  # Special encoding for this file
)

# Add column names manually
df_movies = df_movies.toDF('movie_id', 'year', 'title')

# Show first 10 movies
print("Movie titles:")
df_movies.show(10, truncate=False)
print(f"\nTotal movies: {df_movies.count()}")

Movie titles:
+--------+----+----------------------------+
|movie_id|year|title                       |
+--------+----+----------------------------+
|1       |2003|Dinosaur Planet             |
|2       |2004|Isle of Man TT 2004 Review  |
|3       |1997|Character                   |
|4       |1994|Paula Abdul's Get Up & Dance|
|5       |2004|The Rise and Fall of ECW    |
|6       |1997|Sick                        |
|7       |1992|8 Man                       |
|8       |2004|What the #$*! Do We Know!?  |
|9       |1991|Class of Nuke 'Em High 2    |
|10      |2001|Fighter                     |
+--------+----+----------------------------+
only showing top 10 rows


Total movies: 17770


In [37]:
df_all.createOrReplaceTempView("ratings")


In [43]:
df_movies.createOrReplaceTempView("movies")

AttributeError: 'NoneType' object has no attribute 'show'

In [42]:
spark.sql('select * from ratings limit 5').show()

+--------+-----------+------+-----------+
|movie_id|customer_id|rating|review_date|
+--------+-----------+------+-----------+
|       1|    1488844|     3| 2005-09-06|
|       1|     822109|     5| 2005-05-13|
|       1|     885013|     4| 2005-10-19|
|       1|      30878|     4| 2005-12-26|
|       1|     823519|     3| 2004-05-03|
+--------+-----------+------+-----------+



In [44]:
spark.sql('select * from movies limit 3').show()

+--------+----+--------------------+
|movie_id|year|               title|
+--------+----+--------------------+
|       1|2003|     Dinosaur Planet|
|       2|2004|Isle of Man TT 20...|
|       3|1997|           Character|
+--------+----+--------------------+



In [45]:
result = df_all.join(
    df_movies, 
    on="movie_id",      # Join column
    how="inner"         # Inner join
)

In [46]:
result.createOrReplaceTempView('result')

In [47]:
spark.sql('select * from result limit 2').show(3)

+--------+-----------+------+-----------+----+---------------+
|movie_id|customer_id|rating|review_date|year|          title|
+--------+-----------+------+-----------+----+---------------+
|       1|    1488844|     3| 2005-09-06|2003|Dinosaur Planet|
|       1|     822109|     5| 2005-05-13|2003|Dinosaur Planet|
+--------+-----------+------+-----------+----+---------------+



In [49]:
spark.sql('select movie_id,count(*) from result group by movie_id ').show()

+--------+--------+
|movie_id|count(1)|
+--------+--------+
|       1|     547|
|       3|     308|
|       2|     145|
|    4500|     774|
|    4501|    1785|
|    4503|     129|
|    4502|     312|
+--------+--------+



In [50]:
result.show(5)


+--------+-----------+------+-----------+----+---------------+
|movie_id|customer_id|rating|review_date|year|          title|
+--------+-----------+------+-----------+----+---------------+
|       1|    1488844|     3| 2005-09-06|2003|Dinosaur Planet|
|       1|     822109|     5| 2005-05-13|2003|Dinosaur Planet|
|       1|     885013|     4| 2005-10-19|2003|Dinosaur Planet|
|       1|      30878|     4| 2005-12-26|2003|Dinosaur Planet|
|       1|     823519|     3| 2004-05-03|2003|Dinosaur Planet|
+--------+-----------+------+-----------+----+---------------+
only showing top 5 rows



In [51]:
from pyspark.ml.recommendation import ALS

In [52]:
from pyspark.ml.recommendation import ALS

# Create ALS model
als = ALS(
    userCol="customer_id",      # Which column has user IDs
    itemCol="movie_id",         # Which column has movie IDs
    ratingCol="rating",         # Which column has ratings
    rank=10,                    # Number of hidden features (latent factors)
    maxIter=10,                 # How many iterations to train
    regParam=0.01,              # Regularization (prevent overfitting)
    coldStartStrategy="drop"    # What to do with new users/movies
)

print("ALS model created!")
print(f"Number of hidden features: {als.getRank()}")

ALS model created!
Number of hidden features: 10


In [53]:
print("Training ALS model...")
model = als.fit(result)

Training ALS model...


In [54]:
recommendations = model.recommendForAllUsers(5)


In [55]:
print("Top 5 recommendations for each user:")
recommendations.show(5, truncate=False)

Top 5 recommendations for each user:


+-----------+--------------------------------------------------------------------------------------------+
|customer_id|recommendations                                                                             |
+-----------+--------------------------------------------------------------------------------------------+
|915        |[{1, 4.983452}, {4500, 2.1972642}, {4502, 1.7604702}, {4501, 1.3355701}, {3, -0.56182194}]  |
|989        |[{4501, 1.9931074}, {4503, 1.6262505}, {1, 0.55628186}, {3, 0.5311387}, {4502, 0.014642074}]|
|1333       |[{4503, 4.519964}, {3, 3.9788666}, {4501, 2.9880998}, {2, 1.0628656}, {1, 0.54098207}]      |
|2442       |[{1, 2.9900715}, {4500, 1.3183587}, {4502, 1.0562822}, {4501, 0.8013421}, {3, -0.33709309}] |
|3321       |[{1, 2.9900715}, {4500, 1.3183587}, {4502, 1.0562822}, {4501, 0.8013421}, {3, -0.33709309}] |
+-----------+--------------------------------------------------------------------------------------------+
only showing top 5 rows



In [57]:
# Extract recommendations into a better format
from pyspark.sql.functions import col, explode

# Explode the recommendations array
recs_expanded = recommendations.select(
    "customer_id",
    explode("recommendations").alias("recommendation")
)

# Extract movie_id and rating from the struct
recs_final = recs_expanded.select(
    "customer_id",
    col("recommendation.movie_id").alias("movie_id"),
    col("recommendation.rating").alias("predicted_rating")
)

# Join with movie titles
recs_with_titles = recs_final.join(
    df_movies,
    on="movie_id",
    how="left"
)

print("Recommendations with movie titles:")
recs_with_titles.orderBy("customer_id").show(20, truncate=False)


Recommendations with movie titles:


+--------+-----------+----------------+----+-----------------------------+
|movie_id|customer_id|predicted_rating|year|title                        |
+--------+-----------+----------------+----+-----------------------------+
|1       |915        |4.983452        |2003|Dinosaur Planet              |
|4500    |915        |2.1972642       |1945|Les Dames du Bois de Boulogne|
|4502    |915        |1.7604702       |2001|Do You Wanna Know a Secret?  |
|4501    |915        |1.3355701       |2002|Open Hearts                  |
|3       |915        |-0.56182194     |1997|Character                    |
|4501    |989        |1.9931074       |2002|Open Hearts                  |
|4503    |989        |1.6262505       |1996|Grace of My Heart            |
|1       |989        |0.55628186      |2003|Dinosaur Planet              |
|3       |989        |0.5311387       |1997|Character                    |
|4502    |989        |0.014642074     |2001|Do You Wanna Know a Secret?  |
|4503    |1333       |4.5